# Author: Christina Blackwell
## This notebook postprocesses the DeepSEM results and evaluates the results using VIPER. It also cross references the DeepSEM and GRNBoost2 predictions against the CollecTRI database to determine if any of the predicted TF-target interactions are validated by existing literature.
## CollecTRI GitHub: https://github.com/saezlab/CollecTRI

# Python is used to mount the Google Drive and load the rpy2 module to run R code within the Python environment.

# AI Statement
## During the preparation of this notebook, I used Google Gemini 3 to assist with debugging, determining the correct command line arguments to download the datasets, assisting with identifying correct R syntax, and formatting plots/tables. I subsequently reviewed, edited, and manually verified all logic for accuracy and security.¶

In [28]:
from google.colab import drive
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [29]:
%%R

# Install core managers
if (!requireNamespace("pacman", quietly = TRUE)) install.packages("pacman")
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

# Install decoupleR
BiocManager::install(c("decoupleR", "viper"), update = FALSE, ask = FALSE)

# Load libraries
library(pacman)
library(decoupleR)
library(viper)
p_load(data.table, dplyr, ggpubr, plyr, foreach, parallel, R.utils)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: http://cran.us.r-project.org
Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)
In addition: Warning message:
package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'decoupleR' 'viper' 


In [30]:
%%R
# Initialization parallelism
num.cores <- detectCores()
if(!is.na(num.cores) && (num.cores > 1)) {
  suppressPackageStartupMessages(p_load("doMC"))
  cat(paste("Registering ", num.cores-1, " cores.\n", sep=""))
  registerDoMC(cores=(num.cores-1))
}

Registering 7 cores.


## Download and read datasets from GitHub.

In [31]:
%%R

# Download datasets
sc.data.file <- "CHOOSE-sc-wt-and-tf-log-norm.csv.gz"
sc.metadata.file <- "CHOOSE-sc-wt-and-tf-metadata.csv"
gb2.preds.file <- "postprocessed-grnboost2-all-tfs-celltypes-global-var-genes.csv.gz"
tfs.to.analyze.file <- "CHOOSE-tf-to-analyze-metadata.csv"

# Download datasets
system(paste0("wget -q -O ", sc.data.file, " https://github.com/cblackwe11/TCSS-478-588-Project/raw/main/", sc.data.file))
system(paste0("wget -q -O ", sc.metadata.file, " https://github.com/cblackwe11/TCSS-478-588-Project/raw/main/", sc.metadata.file))
system(paste0("wget -q -O ", gb2.preds.file, " https://github.com/bswhite/morphic-mini-challenge/raw/main/resources/", gb2.preds.file))
system(paste0("wget -q -O ", tfs.to.analyze.file, " https://github.com/bswhite/morphic-mini-challenge/raw/main/resources/", tfs.to.analyze.file))

# Read in CHOOSE scRNA-seq data and metadata
fread_with_rownames <- function(file) {
  tbl <- as.data.frame(fread(file))
  rownames(tbl) <- tbl$V1
  tbl <- tbl[, !(colnames(tbl) == "V1")]
  tbl
}

# Read in tables
sc.data <- fread_with_rownames(sc.data.file)
sc.metadata <- fread_with_rownames(sc.metadata.file)
gb2.preds <- fread_with_rownames(gb2.preds.file)
tfs.to.analyze <- read.csv(tfs.to.analyze.file, header=TRUE)


print(dim(sc.data))
print(dim(sc.metadata))
print(dim(gb2.preds))
print(tfs.to.analyze)


[1] 15685  7338
[1] 7338   75
[1] 654690      9
      TF celltype_jf median.expression.multiome median.expression.scrnaseq
1  MYT1L       ge_in                   1.559380                   1.602277
2 BCL11A     ctx_npc                   1.317189                   1.299130
3  KDM5B       ge_in                   1.688875                   1.598961
4  KMT2A     ctx_npc                   1.266685                   1.213317
5  ASH1L       ge_in                   1.457661                   1.493587
6  ASH1L      ctx_ip                   1.304190                   1.299186
  wt.expr.gt.ko.pval reported.efficiency.of.sgrna
1        0.001133629                       strong
2        0.005426838                       strong
3        0.034082582                       strong
4        0.039354873                       strong
5        0.042533059                       strong
6        0.092750568                       strong


In addition: Warning message:
In fread(file) :
  Detected 75 column names but the data has 76 columns (i.e. invalid file). Added an extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.


In [32]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
%%R
setwd("/content/drive/MyDrive/DeepSEM_Results")

root <- getwd()

# Create resources folder to store plots/tables
output.dir <- paste0(root, "/Analysis_Results/")
dir.create(output.dir, showWarnings = FALSE)

## Read in the GRNBoost2 predictions to later evaluate them with VIPER and compare the results to DeepSEM.

In [34]:
%%R
# The TFs we are targeting for analysis
target_tfs <- unique(tfs.to.analyze$TF)

# Reduce noise, keep only the significant predictions
gb2.preds <- gb2.preds[gb2.preds$significant == TRUE, ]

# Keep only the TFs we are analyzing
gb2.preds <- gb2.preds[gb2.preds$TF %in% target_tfs, ]

print(table(gb2.preds$cell.type, gb2.preds$TF))

         
          ASH1L BCL11A KDM5B KMT2A MYT1L
  ctx_ex    286    291   311   288   325
  ctx_ip    311    260   311   276   333
  ctx_npc   337    352   388   312   235
  ge_in     270    259   318   273   259
  ge_npc    313    312   332   341   219


## This cell postprocesses the results from DeepSEM so it can be evaluated with VIPER. Note that these results were obtained using the original (incorrect) method, where the entire dataset was input to DeepSEM. The resulting dataframe is called `deepsem.preds.global` to reflect this.

## The postprocessing example provided by Dr. White for GRNBoost2 was referenced when writing this code.

##A rank-based cutoff is used to filter DeepSEM, based on the process described by Pratapa et al. in the paper below. They thresholded  results by taking only the top *k* targets. I chose the top 300 MOR scores as the *k* threshold to try to eliminate bias when comparing the DeepSEM VIPER scores to the GRNBoost2 VIPER scores.
## Pratapa, A., Jalihal, A. P., Law, J. N., Bharadwaj, A., & Murali, T. M. (2020). Benchmarking algorithms for gene regulatory network inference from single-cell transcriptomic data. *Nature methods*, 17(2), 147–154. https://doi.org/10.1038/s41592-019-0690-6

In [35]:
%%R

# Read GRN predictions file
net <- as.data.frame(fread("DeepSEM_global_out/GRN_inference_result.tsv"))
colnames(net) <- c("TF", "target", "weight")

# The list of the 5 cell types
cell_types <- c("ctx_ex", "ctx_ip", "ctx_npc", "ge_in", "ge_npc")

# Filter the GRN predictions for just the TFs we are analyzing
net <- net[net$TF %in% target_tfs, ]

# Remove duplicates
unique_genes <- unique(c(net$TF, net$target))

# Filter the network for each cell-type
list_of_dfs <- lapply(cell_types, function(c_type) {
  if(nrow(net) == 0) return(NULL)

  # Get the WT cells for this cell type
  wt.cells <- rownames(sc.metadata[sc.metadata$celltype_jf == c_type & sc.metadata$gRNA_perturb == "CTRL", ])

  # Calculate the correlation
  unique_genes <- unique(c(net$TF, net$target))
  mini_matrix <- t(as.matrix(sc.data[unique_genes, wt.cells]))

  # Suppress warnings for zero-variance genes
  cor_matrix <- suppressWarnings(cor(mini_matrix, method="pearson"))

  # After getting correlation coefficient, sign the edge weights for VIPER
  edge_signs <- sapply(1:nrow(net), function(i) {
    s <- cor_matrix[net$TF[i], net$target[i]]
    if(is.na(s) || s == 0) return(1)
    return(sign(s))
  })

  # Format for VIPER
  net$score <- (net$weight * edge_signs)
  #if(max(abs(net$score)) > 0) net$score <- net$score / max(abs(net$score))
  net$cell.type <- c_type
  net <- net[, c("TF", "target", "score", "cell.type")]

  # Keep the top 300 scores to reduce noise
  # 300 was selected because the GRNBoost2 data had ~300 significant results per TF-cell type pairs
  filtered_tf_list <- lapply(split(net, net$TF), function(df) {
    head(df[order(abs(df$score), decreasing = TRUE), ], 300)
  })

  # Append the filtered TFs together
  return(do.call(rbind, filtered_tf_list))
})

# Append the cell-type specific networks together
deepsem.preds.global <- do.call(rbind, list_of_dfs)

# Scale scores globally
deepsem.preds.global$score <- deepsem.preds.global$score / max(abs(deepsem.preds.global$score))

print(table(deepsem.preds.global$cell.type, deepsem.preds.global$TF))

         
          ASH1L BCL11A KDM5B KMT2A MYT1L
  ctx_ex    300    300   300   300   300
  ctx_ip    300    300   300   300   300
  ctx_npc   300    300   300   300   300
  ge_in     300    300   300   300   300
  ge_npc    300    300   300   300   300


|--------------------------------------------------|
|==================================================|


## This cell postprocesses the results from DeepSEM using the new (correct) method as explained in the `deepSEM_runner.ipynb` notebook.

In [44]:
%%R
# Loop through each cell type, loading and processing its specific TSV
list_of_dfs <- lapply(cell_types, function(c_type) {
    file_path <- paste0(c_type, "/GRN_inference_result.tsv")

    # Read the cell type specific GRN predictions file
    net <- as.data.frame(fread(file_path))
    colnames(net) <- c("TF", "target", "weight")

    # Filter the GRN predictions for just the TFs we are analyzing
    net <- net[net$TF %in% target_tfs, ]
    if(nrow(net) == 0) return(NULL)

    # Get the WT cells for this cell type
    wt.cells <- rownames(sc.metadata[sc.metadata$celltype_jf == c_type & sc.metadata$gRNA_perturb == "CTRL", ])

    # Calculate the correlation
    unique_genes <- unique(c(net$TF, net$target))
    mini_matrix <- t(as.matrix(sc.data[unique_genes, wt.cells]))

    # Suppress warnings for zero-variance genes
    cor_matrix <- suppressWarnings(cor(mini_matrix, method="pearson"))

    # After getting correlation coefficient, sign the edge weights for VIPER
    edge_signs <- sapply(1:nrow(net), function(i) {
      s <- cor_matrix[net$TF[i], net$target[i]]
      if(is.na(s) || s == 0) return(1)
      return(sign(s))
    })

    # Format for VIPER
    net$score <- (net$weight * edge_signs)
    net$cell.type <- c_type
    net <- net[, c("TF", "target", "score", "cell.type")]

    # I do not scale scores globally here because it is a cell-type specific network
    net$score <- net$score / max(abs(net$score))

    # Keep the top 300 scores to reduce noise
    # DeepSEM only outputs edge weights, so it is better to use a rank-based cutoff rather than statistical significance
    # 300 was selected because the GRNBoost2 data had ~300 significant results per TF-cell type pairs
    filtered_tf_list <- lapply(split(net, net$TF), function(df) {
      head(df[order(abs(df$score), decreasing = TRUE), ], 300)
    })

    # Append the filtered TFs together
    return(do.call(rbind, filtered_tf_list))
  })

# Append all the cell-type specific networks together into one master dataframe
deepsem.preds <- do.call(rbind, list_of_dfs)

print(table(deepsem.preds$cell.type, deepsem.preds$TF))

         
          ASH1L BCL11A KDM5B KMT2A MYT1L
  ctx_ex    300    300   300   300   300
  ctx_ip    300    300   300   300   300
  ctx_npc   300    300   300   300   300
  ge_in     300    300   300   300   300
  ge_npc    300    300   300   300   300


|--------------------------------------------------|
|==================================================|
|--------------------------------------------------|
|==================================================|
|--------------------------------------------------|
|==================================================|
|--------------------------------------------------|
|==================================================|
|--------------------------------------------------|
|==================================================|


## VIPER functions, based off Dr. White's `apply-viper-evaluation.R` script.

In [45]:
%%R

score.viper <- function(predictions.tbl, sc.metadata, sc.data,
                        sc.cell.type.col = "celltype_jf", sc.genotype.col = "gRNA_perturb", sc.wt.status = "CTRL")
{
  ddply(predictions.tbl,
        .variables = c("TF", "cell.type"),
        .parallel = FALSE,
        .fun = function(df) {
          tf <- df[1, "TF"]
          ct <- df[1, "cell.type"]

          ko.flag <- (sc.metadata[, sc.genotype.col] == tf) & (sc.metadata[, sc.cell.type.col] == ct)
          wt.flag <- (sc.metadata[, sc.genotype.col] == sc.wt.status) & (sc.metadata[, sc.cell.type.col] == ct)
          ko.mat <- sc.data[, ko.flag]
          wt.mat <- sc.data[, wt.flag]

          network <- data.frame("source" = as.character(tf), "target" = as.character(df$target), "mor" = df$score)
          # VIPER seems to crash if we don't have at least two TFs, so create a dummy.
          dummy.network <- network
          dummy.network$source <- "dummy"
          network <- rbind(network, dummy.network)

          combined.mat <- cbind(ko.mat, wt.mat)
          vp <- run_viper(combined.mat, network, .source = "source", .target = "target", .mor = "mor", pleiotropy = FALSE, method = "none")

          vp <- subset(vp, source != "dummy")
          vp <- merge(vp, sc.metadata[ko.flag | wt.flag, c(sc.genotype.col), drop=FALSE], by.x = c("condition"), by.y = c("row.names"))
          vp <- vp[, !(colnames(vp) %in% c("condition", "statistic", "source", "p_value"))]
          colnames(vp)[colnames(vp) == sc.genotype.col] <- "perturbation"
          vp$condition <- vp$perturbation
          vp$perturbation[vp$perturbation == sc.wt.status] <- "none"
          vp$condition[vp$condition != sc.wt.status] <- "KO"
          vp$condition[vp$condition == sc.wt.status] <- "WT"
          vp <- cbind(vp, TF = tf, cell.type = ct)
          vp
        })
}

compare.viper.distributions <- function(viper.df) {
  stats <-
    ddply(viper.df, .variables = c("cell.type", "TF"),
          .fun = function(df) {
            ret <- wilcox.test(x = subset(df, condition == "WT")$score,
                               y = subset(df, condition == "KO")$score, alternative="greater")
            data.frame(pval = ret$p.value)
          })
  stats <- stats[order(stats$pval, decreasing=FALSE),]
  stats
}

## Calculate VIPER scores and p-values.

In [46]:
%%R

# Run VIPER on the GRNBoost2 networks
gb2.viper <- score.viper(gb2.preds, sc.metadata, sc.data)

# Get the p-values comparing WT vs KO for GRNBoost2
gb2.viper.pvals <- compare.viper.distributions(gb2.viper)

# Run VIPER on the DeepSEM networks
ds.global.viper <- score.viper(deepsem.preds.global, sc.metadata, sc.data)
ds.global.viper.pvals <- compare.viper.distributions(ds.global.viper)

# Get the p-values comparing WT vs KO for DeepSEM networks
ds.viper <- score.viper(deepsem.preds, sc.metadata, sc.data)
ds.viper.pvals <- compare.viper.distributions(ds.viper)

## Create violin plots. This code is based off of the `apply-viper-evaluation.R` script provided by Dr. White. These images are located within `/content/drive/MyDrive/DeepSEM_Results/Analysis_Results`.

In [47]:
%%R

target_pairs <- tfs.to.analyze[, c("TF", "celltype_jf")]
# Rename to match VIPER dataframes
colnames(target_pairs)[2] <- "cell.type"

# Filter the VIPER results to only include the targeted pairs
ds.viper.filtered <- merge(ds.viper, target_pairs, by = c("TF", "cell.type"))
ds.global.viper.filtered <- merge(ds.global.viper, target_pairs, by = c("TF", "cell.type"))
gb2.viper.filtered <- merge(gb2.viper, target_pairs, by = c("TF", "cell.type"))

# Plot DeepSEM (new) results
g_ds_new <- ggviolin(data = ds.viper.filtered,
                 x = "condition",
                 y = "score",
                 fill = "white", color = "black", trim = FALSE) +

  stat_compare_means(aes(label = sprintf("p = %.5f", ..p..)),
                     method = "wilcox.test",
                     method.args = list(alternative = "greater"),
                     ref.group = "KO",
                     label.x.npc = "left", label.y.npc = "top",
                     hjust = -0.1, size = 4) +

  facet_wrap(~ cell.type + TF, scales = "free_y", ncol = 3) +

  ggtitle("DeepSEM: Targeted TF Activity Scores (Cell-Type)") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    strip.background = element_rect(fill = "#f2f2f2", color = "black"),
    strip.text = element_text(color = "black", size = 12)
  )

ofile_ds <- paste0(output.dir, "CHOOSE-TF-Viper-DeepSEM-Cell-Type-Targeted.png")
png(ofile_ds, width = 2000, height = 1600, res = 150)
print(g_ds_new)
dev.off()

# Plot DeepSEM (global) results
g_ds <- ggviolin(data = ds.global.viper.filtered,
                 x = "condition",
                 y = "score",
                 fill = "white", color = "black", trim = FALSE) +

  stat_compare_means(aes(label = sprintf("p = %.5f", ..p..)),
                     method = "wilcox.test",
                     method.args = list(alternative = "greater"),
                     ref.group = "KO",
                     label.x.npc = "left", label.y.npc = "top",
                     hjust = -0.1, size = 4) +

  facet_wrap(~ cell.type + TF, scales = "free_y", ncol = 3) +

  ggtitle("DeepSEM: Targeted TF Activity Scores (Global)") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    strip.background = element_rect(fill = "#f2f2f2", color = "black"),
    strip.text = element_text(color = "black", size = 12)
  )

ofile_ds <- paste0(output.dir, "CHOOSE-TF-Viper-DeepSEM-Global-Targeted.png")
png(ofile_ds, width = 2000, height = 1600, res = 150)
print(g_ds)
dev.off()

# Plot GRNBoost2 results
g_gb2 <- ggviolin(data = gb2.viper.filtered,
                  x = "condition",
                  y = "score",
                  fill = "white", color = "black", trim = FALSE) +

  stat_compare_means(aes(label = sprintf("p = %.5f", ..p..)),
                     method = "wilcox.test",
                     method.args = list(alternative = "greater"),
                     ref.group = "KO",
                     label.x.npc = "left", label.y.npc = "top",
                     hjust = -0.1, size = 4) +

  facet_wrap(~ cell.type + TF, scales = "free_y", ncol = 3) +

  ggtitle("GRNBoost2: Targeted TF Activity Scores") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    strip.background = element_rect(fill = "#f2f2f2", color = "black"),
    strip.text = element_text(color = "black", size = 12)
  )

ofile_gb2 <- paste0(output.dir, "CHOOSE-TF-Viper-GRNBoost2-Targeted.png")
png(ofile_gb2, width = 2000, height = 1600, res = 150)
print(g_gb2)
dev.off()


png 
  2 


## Create a summary table, indicating the winners (the method with the lowest p-value per pair). Located under `/content/drive/MyDrive/DeepSEM_Results/Analysis_Results`.

In [49]:
%%R

# Update column names to distinguish the three models
colnames(ds.global.viper.pvals)[colnames(ds.global.viper.pvals) == "pval"] <- "pval_DeepSEM_Global"
colnames(ds.viper.pvals)[colnames(ds.viper.pvals) == "pval"] <- "pval_DeepSEM_CellType"
colnames(gb2.viper.pvals)[colnames(gb2.viper.pvals) == "pval"] <- "pval_GRNBoost2"

# Merge all three tables together
p_list <- list(ds.global.viper.pvals, ds.viper.pvals, gb2.viper.pvals)
summary_table <- Reduce(function(x, y) merge(x, y, by = c("cell.type", "TF"), all = TRUE), p_list)

# Determine which of the models produced the most significant p-value
summary_table$Winner <- apply(summary_table[, c("pval_DeepSEM_Global", "pval_DeepSEM_CellType", "pval_GRNBoost2")], 1, function(x) {
  if(all(is.na(x))) return("None")
  models <- c("DeepSEM_Global", "DeepSEM_CellType", "GRNBoost2")
  return(models[which.min(x)])
})

# Determine if the new DeepSEM results method beat the global results
summary_table$CellType_Improved <- ifelse(
  summary_table$pval_DeepSEM_CellType < summary_table$pval_DeepSEM_Global,
  "Yes", "No"
)

# Flag targeted pairs (the pairs we are focusing our analysis on)
target_pairs$Targeted_Pair <- "Yes"
summary_table <- merge(summary_table, target_pairs, by = c("cell.type", "TF"), all.x = TRUE)
summary_table$Targeted_Pair[is.na(summary_table$Targeted_Pair)] <- "No"

# Sort the table so targeted pairs appear first, then by best DeepSEM results
summary_table <- summary_table[order(
  factor(summary_table$Targeted_Pair, levels = c("Yes", "No")),
  pmin(summary_table$pval_DeepSEM_Global, summary_table$pval_DeepSEM_CellType, na.rm = TRUE),
  decreasing = FALSE
), ]

# Print and save
print("Comparison Summary Table:")
print(head(summary_table, 20), row.names = FALSE)

summary_file <- paste0(output.dir, "DeepSEM_vs_GRNBoost2_Summary.csv")
write.csv(summary_table, file = summary_file, row.names = FALSE)

[1] "Comparison Summary Table:"
 cell.type     TF pval_DeepSEM_Global pval_DeepSEM_CellType pval_GRNBoost2
     ge_in  MYT1L        9.928987e-05           0.122642596    0.735104979
   ctx_npc BCL11A        3.748351e-04           0.009594034    0.028275977
   ctx_npc  KMT2A        7.798296e-01           0.533804741    0.957005938
     ge_in  KDM5B        7.636914e-01           0.584950310    0.594220646
    ctx_ip  ASH1L        6.133898e-01           0.633170663    0.549880706
     ge_in  ASH1L        8.608997e-01           0.733270878    0.788101902
   ctx_npc  KDM5B        1.605415e-02           0.333594055    0.375190228
    ctx_ex  MYT1L        4.442976e-02           0.676094203    0.006896116
    ctx_ip BCL11A        5.934509e-02           0.240663103    0.980681842
    ge_npc  MYT1L        9.571579e-02           0.206931676    0.752732610
    ge_npc  ASH1L        2.692094e-01           0.113894402    0.745431882
    ctx_ip  MYT1L        1.593924e-01           0.525118653    0.410

## Validate results with CollecTRI database.

In [53]:
%%R

# Download the CollecTRI Database
api_url <- "https://omnipathdb.org/interactions?datasets=collectri&format=tsv&genesymbols=1"
collectri <- read.table(api_url, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
collectri_edges <- unique(paste(collectri$source_genesymbol, collectri$target_genesymbol, sep = "_"))

# Extract unique edges from all three models (filter out repeats if they exist)
ds_new_edges <- unique(paste(deepsem.preds$TF, deepsem.preds$target, sep = "_"))
ds_global_edges <- unique(paste(deepsem.preds.global$TF, deepsem.preds.global$target, sep = "_"))
gb2_edges <- unique(paste(gb2.preds$TF, gb2.preds$target, sep = "_"))

# Calculate stats and extract validated links
validate_model <- function(model_edges, model_name) {
  # Find which edges exist in CollecTRI
  hits_mask <- model_edges %in% collectri_edges
  validated_links <- model_edges[hits_mask]

  hits_count <- length(validated_links)

  # What percentage of the model's predictions (after filtering) were validated by CollecTRI
  precision <- (hits_count / length(model_edges)) * 100

  cat(sprintf("%-18s: %d validated edges (%.2f%% Precision)\n",
              model_name, hits_count, precision))

  return(validated_links)
}

# Run validation and store the specific validated gene links
valid_ds_new <- validate_model(ds_new_edges, "DeepSEM (Cell-Type)")
valid_ds_global <- validate_model(ds_global_edges, "DeepSEM (Global)")
valid_gb2 <- validate_model(gb2_edges, "GRNBoost2")

cat("\nValidated Links in Cell-Type DeepSEM Results:\n")
print(valid_ds_new)

cat("\nValidated Links in Global DeepSEM Results:\n")
print(valid_ds_global)

cat("\nValidated Links in GRNBoost2 Results:\n")
print(valid_gb2)



DeepSEM (Cell-Type): 13 validated edges (0.18% Precision)
DeepSEM (Global)  : 6 validated edges (0.40% Precision)
GRNBoost2         : 3 validated edges (0.04% Precision)

Validated Links in Cell-Type DeepSEM Results:
 [1] "KDM5B_FOXG1"   "KDM5B_BMI1"    "KMT2A_MEIS1"   "KMT2A_HLA-E"  
 [5] "KMT2A_CHD8"    "KMT2A_KMT2B"   "KDM5B_TGFB2"   "KMT2A_MAGOHB" 
 [9] "MYT1L_YAP1"    "BCL11A_TCF4"   "BCL11A_DCC"    "KMT2A_SMARCA2"
[13] "KMT2A_ABCB1"  

Validated Links in Global DeepSEM Results:
[1] "BCL11A_DCC"   "BCL11A_TCF4"  "BCL11A_MAP1B" "KMT2A_JUN"    "KMT2A_ASXL1" 
[6] "KMT2A_CDKN1B"

Validated Links in GRNBoost2 Results:
[1] "KDM5B_CDKN1A" "KMT2A_FGFR1"  "BCL11A_DCC"  
